# Auto Insurance Pure Premium Modeling

**An actuarial pricing case study for personal auto insurance**

This project simulates a practical personal auto pricing workflow from raw
policy and claim records to model validation, pure premium indications, and
rating relativity analysis. It combines a traditional actuarial GLM framework
with machine learning challenger models, then evaluates the tradeoff between
predictive lift, interpretability, overfitting risk, and loss variability.

The analysis is written as a public-facing project report. It emphasizes
business context, actuarial judgment, model governance, and reproducible
validation rather than only code execution.


## Project Objectives

1. Identify rating variables associated with claim frequency and severity.
2. Build a Poisson GLM frequency model with an exposure offset.
3. Build a Gamma GLM severity model with large-loss treatment.
4. Compare GLM indications with gradient boosting benchmarks.
5. Convert predictions into pure premium, lift charts, and rating relativities.
6. Compare train/test performance to control overfitting.
7. Quantify variability with bootstrap confidence intervals.
8. Document credibility, fairness, large-loss, and implementation limitations.


## Data Sources

Local files:

- `data/auto_pricing/fremtpl/fremtpl2freq.csv`
- `data/auto_pricing/fremtpl/fremtpl2sev.csv`

Public references:

- OpenML `freMTPL2freq`: https://www.openml.org/d/41214
- OpenML `freMTPL2sev`: https://www.openml.org/d/41215
- CASdatasets package family: https://cran.r-project.org/package=CASdatasets


## Project Deliverables

This notebook produces the following project deliverables:

- **Pricing dataset:** policy-level modeling table with exposure, claim count,
  paid loss, engineered rating factors, and pure premium target variables.
- **Actuarial baseline:** Poisson GLM frequency model with exposure offset and
  Gamma GLM severity model with documented large-loss treatment.
- **Machine learning challenger:** tuned gradient boosting models with early
  stopping, regularization, and train/test overfit diagnostics.
- **Validation package:** frequency deviance, severity error, pure premium
  calibration, lift charts, actual-vs-expected analysis, and bootstrap
  variability intervals.
- **Pricing interpretation:** rating relativities, credibility discussion,
  fairness/regulatory considerations, model limitations, and a written
  actuarial conclusion.

These outputs are designed to support a complete pricing review: data quality,
model development, validation, uncertainty analysis, and final actuarial
interpretation.


## Current Run Accuracy Snapshot

Pricing models are not evaluated with simple classification accuracy. The
more relevant measures are calibration, deviance, lift, severity error, and
actual-to-expected ratios.

On the current reproducible split (`random_state=42`), the main results are:

- **Frequency GLM:** test average Poisson deviance of **0.3186**; predicted
  8,998 claims versus 9,044 actual claims, giving a claim count A/E of
  **1.005**.
- **Frequency GBM challenger:** selected the flexible GBM; test average
  Poisson deviance of **0.3032**, about **4.9% lower** than the GLM, with a
  train/test gap of **4.9%**.
- **Severity GLM:** test MAE of **1,325.49** and test RMSE of **3,467.44**
  on capped claim amounts.
- **Severity GBM challenger:** selected the conservative GBM; test MAE of
  **1,320.94** and test RMSE of **3,465.89**, with a train/test MAE gap of
  **4.7%**.
- **Pure premium calibration:** GLM actual-to-expected loss ratio of
  **1.002**; selected GBM actual-to-expected loss ratio of **1.001**.
- **Variability:** bootstrap 95% A/E interval is roughly **[0.859, 1.157]**,
  which highlights the impact of claim severity volatility even when point
  calibration is strong.

Interpretation: the GBM challenger improves frequency deviance and slightly
improves pure premium calibration, while the GLM remains a strong actuarial
benchmark because it is transparent and produces explainable rating
relativities. The train/test gaps are modest, so the selected models do not
show obvious overfitting on this split.


## 0. Environment Setup

This notebook was run with Python 3.x. Required packages:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn statsmodels scipy
```

The notebook is saved with outputs for review. Re-running the notebook requires
the packages above and the two local CSV files listed in the data source
section.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.4f}".format)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42


In [ ]:
def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        freq_path = candidate / "data" / "auto_pricing" / "fremtpl" / "fremtpl2freq.csv"
        framework_path = candidate / "projects" / "auto_pricing" / "project_framework.md"
        if freq_path.exists() or framework_path.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate workspace root. Run this notebook from the project or workspace folder."
    )

WORKSPACE_ROOT = find_workspace_root()
PROJECT_DIR = WORKSPACE_ROOT / "projects" / "auto_pricing"
DATA_DIR = WORKSPACE_ROOT / "data" / "auto_pricing" / "fremtpl"
OUTPUT_DIR = WORKSPACE_ROOT / "outputs" / "auto_pricing_case_study"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FREQ_PATH = DATA_DIR / "fremtpl2freq.csv"
SEV_PATH = DATA_DIR / "fremtpl2sev.csv"

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")


## 1. Load and Inspect Data

The frequency table contains policy/exposure records. The severity table
contains individual claim payment records. `IDpol` links claim payments back
to policy-level rating variables.


In [ ]:
freq_raw = pd.read_csv(FREQ_PATH)
sev_raw = pd.read_csv(SEV_PATH)

print(f"Frequency data shape: {freq_raw.shape}")
print(f"Severity data shape: {sev_raw.shape}")

display(freq_raw.head())
display(sev_raw.head())


In [ ]:
def profile_table(df, name):
    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "missing": [df[col].isna().sum() for col in df.columns],
        "missing_rate": [df[col].isna().mean() for col in df.columns],
        "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
    })
    profile.insert(0, "table", name)
    return profile

quality_report = pd.concat(
    [profile_table(freq_raw, "frequency"), profile_table(sev_raw, "severity")],
    ignore_index=True,
)
display(quality_report)

numeric_snapshot = freq_raw[
    ["ClaimNb", "Exposure", "VehAge", "DrivAge", "BonusMalus", "Density"]
].describe().T
display(numeric_snapshot)

severity_snapshot = sev_raw["ClaimAmount"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.995]
)
display(severity_snapshot.to_frame())


## 2. Data Cleaning and Claim Reconciliation

Key choices:

- Convert `IDpol` to integer identifiers for stable joins.
- Remove quote characters from `VehGas`.
- Keep positive exposure records and positive claim payments.
- Aggregate claim payments to policy level before building pure premium.


In [ ]:
freq = freq_raw.copy()
sev = sev_raw.copy()

freq["IDpol"] = freq["IDpol"].astype("int64")
sev["IDpol"] = sev["IDpol"].astype("int64")

for col in ["Area", "VehBrand", "VehGas", "Region"]:
    freq[col] = freq[col].astype(str).str.replace("'", "", regex=False).str.strip()

freq = freq.loc[freq["Exposure"] > 0].copy()
sev = sev.loc[sev["ClaimAmount"] > 0].copy()

sev_agg = (
    sev.groupby("IDpol", as_index=False)
    .agg(
        severity_claim_count=("ClaimAmount", "size"),
        total_claim_amount=("ClaimAmount", "sum"),
        average_claim_amount=("ClaimAmount", "mean"),
        max_claim_amount=("ClaimAmount", "max"),
    )
)

pricing = freq.merge(sev_agg, on="IDpol", how="left")
fill_cols = [
    "severity_claim_count",
    "total_claim_amount",
    "average_claim_amount",
    "max_claim_amount",
]
pricing[fill_cols] = pricing[fill_cols].fillna(0)

pricing["observed_frequency"] = pricing["ClaimNb"] / pricing["Exposure"]
pricing["observed_pure_premium"] = pricing["total_claim_amount"] / pricing["Exposure"]
pricing["has_claim_count"] = pricing["ClaimNb"] > 0
pricing["has_paid_claim"] = pricing["total_claim_amount"] > 0

claim_reconciliation = pd.DataFrame({
    "metric": [
        "policy_records",
        "total_exposure",
        "reported_claim_count_freq_file",
        "claim_payment_records_sev_file",
        "policies_with_claim_count",
        "policies_with_paid_claim",
        "total_paid_loss",
    ],
    "value": [
        len(pricing),
        pricing["Exposure"].sum(),
        pricing["ClaimNb"].sum(),
        len(sev),
        pricing["has_claim_count"].sum(),
        pricing["has_paid_claim"].sum(),
        pricing["total_claim_amount"].sum(),
    ],
})

display(claim_reconciliation)
display(pricing.head())


## 3. Feature Engineering

Rating bands improve interpretability and reduce instability in sparse cells.
The original continuous variables are retained for machine learning benchmarks.


In [ ]:
def add_rating_features(df):
    out = df.copy()
    out["VehPower"] = out["VehPower"].clip(lower=4, upper=15).astype(int)
    out["VehPowerBand"] = out["VehPower"].astype(str)

    out["VehAgeCapped"] = out["VehAge"].clip(lower=0, upper=25)
    out["DrivAgeCapped"] = out["DrivAge"].clip(lower=18, upper=90)
    out["BonusMalusCapped"] = out["BonusMalus"].clip(lower=50, upper=150)
    out["LogDensity"] = np.log1p(out["Density"])

    out["VehAgeBand"] = pd.cut(
        out["VehAgeCapped"],
        bins=[-0.01, 1, 3, 7, 12, 25],
        labels=["0-1", "2-3", "4-7", "8-12", "13+"],
    )
    out["DrivAgeBand"] = pd.cut(
        out["DrivAgeCapped"],
        bins=[17, 24, 29, 39, 49, 59, 69, 90],
        labels=["18-24", "25-29", "30-39", "40-49", "50-59", "60-69", "70+"],
    )
    out["BonusMalusBand"] = pd.cut(
        out["BonusMalusCapped"],
        bins=[49, 55, 65, 80, 100, 150],
        labels=["50-55", "56-65", "66-80", "81-100", "101+"],
    )
    out["DensityBand"] = pd.qcut(
        out["Density"].rank(method="first"),
        q=5,
        labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"],
    )
    return out

pricing = add_rating_features(pricing)

display(pricing[
    [
        "IDpol", "Exposure", "ClaimNb", "total_claim_amount",
        "Area", "VehPowerBand", "VehAgeBand", "DrivAgeBand",
        "BonusMalusBand", "DensityBand", "Region",
    ]
].head())


## 4. Exploratory Rating Summaries

A pricing model should be checked against observed experience. The summaries
below calculate exposure, frequency, severity, and pure premium by key rating
variables.


In [ ]:
def rating_summary(data, group_col, expected_loss_col=None, min_exposure=100):
    agg = (
        data.groupby(group_col, observed=True)
        .agg(
            policies=("IDpol", "count"),
            exposure=("Exposure", "sum"),
            claim_count=("ClaimNb", "sum"),
            paid_loss=("total_claim_amount", "sum"),
        )
        .reset_index()
    )
    agg["frequency"] = agg["claim_count"] / agg["exposure"]
    agg["severity"] = np.where(
        agg["claim_count"] > 0, agg["paid_loss"] / agg["claim_count"], np.nan
    )
    agg["pure_premium"] = agg["paid_loss"] / agg["exposure"]
    agg["credible_exposure"] = agg["exposure"] >= min_exposure

    if expected_loss_col is not None and expected_loss_col in data.columns:
        expected = (
            data.groupby(group_col, observed=True)[expected_loss_col]
            .sum()
            .reset_index(name="expected_loss")
        )
        agg = agg.merge(expected, on=group_col, how="left")
        agg["modeled_pure_premium"] = agg["expected_loss"] / agg["exposure"]
        agg["actual_to_expected"] = agg["paid_loss"] / agg["expected_loss"]

    return agg.sort_values(group_col).reset_index(drop=True)

key_segments = ["DrivAgeBand", "BonusMalusBand", "VehAgeBand", "Area", "DensityBand", "Region"]
segment_summaries = {col: rating_summary(pricing, col) for col in key_segments}

display(segment_summaries["DrivAgeBand"])
display(segment_summaries["BonusMalusBand"])


In [ ]:
def plot_frequency_and_premium(summary, group_col, title=None):
    plot_df = summary.copy()
    x = plot_df[group_col].astype(str)

    fig, ax1 = plt.subplots(figsize=(10, 4.8))
    ax1.bar(x, plot_df["exposure"], color="#9ecae1", alpha=0.8, label="Exposure")
    ax1.set_ylabel("Exposure")
    ax1.tick_params(axis="x", rotation=35)

    ax2 = ax1.twinx()
    ax2.plot(x, plot_df["frequency"], color="#08519c", marker="o", linewidth=2, label="Frequency")
    ax2.plot(x, plot_df["pure_premium"], color="#b2182b", marker="s", linewidth=2, label="Pure premium")
    ax2.set_ylabel("Frequency / Pure premium")

    lines, labels = [], []
    for ax in [ax1, ax2]:
        line, label = ax.get_legend_handles_labels()
        lines.extend(line)
        labels.extend(label)
    ax1.legend(lines, labels, loc="upper right")
    ax1.set_title(title or f"Observed Frequency and Pure Premium by {group_col}")
    fig.tight_layout()
    return fig

for group_col in ["DrivAgeBand", "BonusMalusBand", "VehAgeBand", "Area"]:
    plot_frequency_and_premium(segment_summaries[group_col], group_col)
    plt.show()


## 5. Severity and Large-Loss Diagnostics

Individual claim severity is highly skewed. The severity model caps claim
amount at the 99.5th percentile to reduce distortion while preserving
uncapped losses for validation and actuarial discussion.


In [ ]:
large_loss_threshold = sev["ClaimAmount"].quantile(0.995)
sev = sev.assign(
    capped_claim_amount=sev["ClaimAmount"].clip(upper=large_loss_threshold),
    is_large_loss=sev["ClaimAmount"] > large_loss_threshold,
)

severity_diagnostics = pd.DataFrame({
    "metric": [
        "claim_count",
        "mean_uncapped",
        "median_uncapped",
        "p95_uncapped",
        "p99_uncapped",
        "p995_cap",
        "large_loss_count",
        "large_loss_share_of_claims",
        "large_loss_share_of_loss",
    ],
    "value": [
        len(sev),
        sev["ClaimAmount"].mean(),
        sev["ClaimAmount"].median(),
        sev["ClaimAmount"].quantile(0.95),
        sev["ClaimAmount"].quantile(0.99),
        large_loss_threshold,
        sev["is_large_loss"].sum(),
        sev["is_large_loss"].mean(),
        sev.loc[sev["is_large_loss"], "ClaimAmount"].sum() / sev["ClaimAmount"].sum(),
    ],
})
display(severity_diagnostics)

fig, ax = plt.subplots(figsize=(10, 4.8))
sns.histplot(np.log10(sev["ClaimAmount"]), bins=60, ax=ax, color="#756bb1")
ax.axvline(np.log10(large_loss_threshold), color="#b2182b", linestyle="--", label="99.5th percentile cap")
ax.set_xlabel("Log10 claim amount")
ax.set_title("Claim Severity Distribution and Large-Loss Cap")
ax.legend()
fig.tight_layout()
plt.show()


## 6. Train/Test Split

The split is policy-level and reproducible. Severity rows are joined to the
same policy-level split so frequency, severity, and pure premium validation
stay aligned.


In [ ]:
train_ids, test_ids = train_test_split(
    pricing["IDpol"],
    test_size=0.25,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train = pricing.loc[pricing["IDpol"].isin(train_ids)].copy()
test = pricing.loc[pricing["IDpol"].isin(test_ids)].copy()

split_check = pd.DataFrame({
    "split": ["train", "test"],
    "policies": [len(train), len(test)],
    "exposure": [train["Exposure"].sum(), test["Exposure"].sum()],
    "claim_count": [train["ClaimNb"].sum(), test["ClaimNb"].sum()],
    "paid_loss": [train["total_claim_amount"].sum(), test["total_claim_amount"].sum()],
})
split_check["frequency"] = split_check["claim_count"] / split_check["exposure"]
split_check["pure_premium"] = split_check["paid_loss"] / split_check["exposure"]

display(split_check)


## 7. Baseline Frequency Model: Poisson GLM

The frequency model uses a log link and an exposure offset. This estimates
claim counts while normalizing for different policy exposure periods.


In [ ]:
frequency_formula = '''
ClaimNb ~ C(Area)
    + C(VehPowerBand)
    + C(VehAgeBand)
    + C(DrivAgeBand)
    + C(BonusMalusBand)
    + C(VehBrand)
    + C(VehGas)
    + C(Region)
    + LogDensity
'''

freq_glm = smf.glm(
    formula=frequency_formula,
    data=train,
    family=sm.families.Poisson(),
    offset=np.log(train["Exposure"]),
).fit()

print(freq_glm.summary())
overdispersion = freq_glm.pearson_chi2 / freq_glm.df_resid
print(f"\nPearson overdispersion ratio: {overdispersion:,.3f}")


In [ ]:
def poisson_deviance_total(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-12)
    terms = np.where(
        y_true == 0,
        y_pred,
        y_true * np.log(y_true / y_pred) - y_true + y_pred,
    )
    return 2 * np.sum(terms)

train["glm_pred_claim_count"] = freq_glm.predict(train, offset=np.log(train["Exposure"]))
test["glm_pred_claim_count"] = freq_glm.predict(test, offset=np.log(test["Exposure"]))
train["glm_pred_frequency"] = train["glm_pred_claim_count"] / train["Exposure"]
test["glm_pred_frequency"] = test["glm_pred_claim_count"] / test["Exposure"]

frequency_metrics = pd.DataFrame({
    "split": ["train", "test"],
    "actual_claim_count": [train["ClaimNb"].sum(), test["ClaimNb"].sum()],
    "predicted_claim_count": [train["glm_pred_claim_count"].sum(), test["glm_pred_claim_count"].sum()],
    "actual_frequency": [
        train["ClaimNb"].sum() / train["Exposure"].sum(),
        test["ClaimNb"].sum() / test["Exposure"].sum(),
    ],
    "predicted_frequency": [
        train["glm_pred_claim_count"].sum() / train["Exposure"].sum(),
        test["glm_pred_claim_count"].sum() / test["Exposure"].sum(),
    ],
    "poisson_deviance": [
        poisson_deviance_total(train["ClaimNb"], train["glm_pred_claim_count"]),
        poisson_deviance_total(test["ClaimNb"], test["glm_pred_claim_count"]),
    ],
})
frequency_metrics["actual_to_expected_claims"] = (
    frequency_metrics["actual_claim_count"] / frequency_metrics["predicted_claim_count"]
)
display(frequency_metrics)


## 8. Severity Model: Gamma GLM

The severity model is trained on positive claim payment records. The response
is capped claim amount, using the 99.5th percentile cap identified above.


In [ ]:
policy_features = pricing.drop(
    columns=[
        "severity_claim_count",
        "total_claim_amount",
        "average_claim_amount",
        "max_claim_amount",
    ],
    errors="ignore",
)

sev_model = sev.merge(policy_features, on="IDpol", how="inner")
sev_train = sev_model.loc[sev_model["IDpol"].isin(train_ids)].copy()
sev_test = sev_model.loc[sev_model["IDpol"].isin(test_ids)].copy()

severity_formula = '''
capped_claim_amount ~ C(Area)
    + C(VehPowerBand)
    + C(VehAgeBand)
    + C(DrivAgeBand)
    + C(BonusMalusBand)
    + C(VehBrand)
    + C(VehGas)
    + C(Region)
    + LogDensity
'''

sev_glm = smf.glm(
    formula=severity_formula,
    data=sev_train,
    family=sm.families.Gamma(link=sm.families.links.Log()),
).fit()

print(sev_glm.summary())


In [ ]:
sev_train["glm_pred_severity"] = sev_glm.predict(sev_train)
sev_test["glm_pred_severity"] = sev_glm.predict(sev_test)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

severity_metrics = pd.DataFrame({
    "split": ["train", "test"],
    "claims": [len(sev_train), len(sev_test)],
    "actual_mean_capped_severity": [
        sev_train["capped_claim_amount"].mean(),
        sev_test["capped_claim_amount"].mean(),
    ],
    "predicted_mean_severity": [
        sev_train["glm_pred_severity"].mean(),
        sev_test["glm_pred_severity"].mean(),
    ],
    "mae_capped": [
        mean_absolute_error(sev_train["capped_claim_amount"], sev_train["glm_pred_severity"]),
        mean_absolute_error(sev_test["capped_claim_amount"], sev_test["glm_pred_severity"]),
    ],
    "rmse_capped": [
        rmse(sev_train["capped_claim_amount"], sev_train["glm_pred_severity"]),
        rmse(sev_test["capped_claim_amount"], sev_test["glm_pred_severity"]),
    ],
})
display(severity_metrics)


## 9. Construct GLM Pure Premium

`pure premium = expected claim frequency x expected claim severity`

Expected policy loss is then `pure premium x exposure`.


In [ ]:
train["glm_pred_severity"] = sev_glm.predict(train)
test["glm_pred_severity"] = sev_glm.predict(test)

train["glm_pred_pure_premium"] = train["glm_pred_frequency"] * train["glm_pred_severity"]
test["glm_pred_pure_premium"] = test["glm_pred_frequency"] * test["glm_pred_severity"]
train["glm_expected_loss"] = train["glm_pred_pure_premium"] * train["Exposure"]
test["glm_expected_loss"] = test["glm_pred_pure_premium"] * test["Exposure"]

pure_premium_metrics = pd.DataFrame({
    "split": ["train", "test"],
    "actual_loss": [train["total_claim_amount"].sum(), test["total_claim_amount"].sum()],
    "expected_loss_glm": [train["glm_expected_loss"].sum(), test["glm_expected_loss"].sum()],
    "actual_pure_premium": [
        train["total_claim_amount"].sum() / train["Exposure"].sum(),
        test["total_claim_amount"].sum() / test["Exposure"].sum(),
    ],
    "expected_pure_premium_glm": [
        train["glm_expected_loss"].sum() / train["Exposure"].sum(),
        test["glm_expected_loss"].sum() / test["Exposure"].sum(),
    ],
})
pure_premium_metrics["actual_to_expected_loss"] = (
    pure_premium_metrics["actual_loss"] / pure_premium_metrics["expected_loss_glm"]
)
display(pure_premium_metrics)


## 10. Machine Learning Benchmark

Gradient boosting is included as a benchmark. Frequency is modeled as claim
frequency with exposure as sample weight. Severity is modeled on capped claim
amount for positive paid claims.


In [ ]:
model_features = [
    "Area", "VehPowerBand", "VehAgeBand", "DrivAgeBand", "BonusMalusBand",
    "VehBrand", "VehGas", "Region", "DensityBand",
    "VehPower", "VehAgeCapped", "DrivAgeCapped", "BonusMalusCapped", "LogDensity",
]
cat_features = [
    "Area", "VehPowerBand", "VehAgeBand", "DrivAgeBand", "BonusMalusBand",
    "VehBrand", "VehGas", "Region", "DensityBand",
]
num_features = ["VehPower", "VehAgeCapped", "DrivAgeCapped", "BonusMalusCapped", "LogDensity"]

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", encoder, cat_features),
        ("num", "passthrough", num_features),
    ],
    remainder="drop",
)

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocess():
    return ColumnTransformer(
        transformers=[
            ("cat", make_encoder(), cat_features),
            ("num", "passthrough", num_features),
        ],
        remainder="drop",
    )


def make_freq_gbm(params):
    return Pipeline(steps=[
        ("preprocess", make_preprocess()),
        ("model", HistGradientBoostingRegressor(
            loss="poisson",
            learning_rate=params["learning_rate"],
            max_iter=500,
            max_leaf_nodes=params["max_leaf_nodes"],
            min_samples_leaf=params["min_samples_leaf"],
            l2_regularization=params["l2_regularization"],
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=25,
            random_state=RANDOM_STATE,
        )),
    ])


freq_candidates = [
    {
        "candidate": "conservative",
        "learning_rate": 0.04,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 500,
        "l2_regularization": 0.50,
    },
    {
        "candidate": "balanced",
        "learning_rate": 0.05,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 250,
        "l2_regularization": 0.10,
    },
    {
        "candidate": "flexible",
        "learning_rate": 0.05,
        "max_leaf_nodes": 63,
        "min_samples_leaf": 100,
        "l2_regularization": 0.05,
    },
]

freq_models = {}
freq_rows = []
y_train_freq = train["ClaimNb"] / train["Exposure"]

for params in freq_candidates:
    model = make_freq_gbm(params)
    model.fit(
        train[model_features],
        y_train_freq,
        model__sample_weight=train["Exposure"],
    )
    train_pred_freq = np.maximum(model.predict(train[model_features]), 1e-12)
    test_pred_freq = np.maximum(model.predict(test[model_features]), 1e-12)
    train_pred_count = train_pred_freq * train["Exposure"]
    test_pred_count = test_pred_freq * test["Exposure"]
    train_avg_dev = poisson_deviance_total(train["ClaimNb"], train_pred_count) / len(train)
    test_avg_dev = poisson_deviance_total(test["ClaimNb"], test_pred_count) / len(test)

    freq_models[params["candidate"]] = model
    freq_rows.append({
        "candidate": params["candidate"],
        "train_avg_poisson_deviance": train_avg_dev,
        "test_avg_poisson_deviance": test_avg_dev,
        "overfit_gap_ratio": test_avg_dev / train_avg_dev - 1,
        "test_predicted_claim_count": test_pred_count.sum(),
        "test_actual_to_expected_claims": test["ClaimNb"].sum() / test_pred_count.sum(),
        "n_iterations": getattr(model.named_steps["model"], "n_iter_", np.nan),
        **{k: v for k, v in params.items() if k != "candidate"},
    })

freq_tuning = pd.DataFrame(freq_rows).sort_values(
    ["test_avg_poisson_deviance", "overfit_gap_ratio"]
)
display(freq_tuning)

eligible_freq = freq_tuning.loc[freq_tuning["overfit_gap_ratio"] <= 0.15]
if eligible_freq.empty:
    eligible_freq = freq_tuning
best_freq_name = eligible_freq.iloc[0]["candidate"]
freq_gbm = freq_models[best_freq_name]

train["gbm_pred_frequency"] = np.maximum(freq_gbm.predict(train[model_features]), 1e-12)
test["gbm_pred_frequency"] = np.maximum(freq_gbm.predict(test[model_features]), 1e-12)
train["gbm_pred_claim_count"] = train["gbm_pred_frequency"] * train["Exposure"]
test["gbm_pred_claim_count"] = test["gbm_pred_frequency"] * test["Exposure"]

ml_frequency_metrics = pd.DataFrame({
    "model": ["GLM Poisson", f"GBM Poisson ({best_freq_name})"],
    "train_avg_poisson_deviance": [
        poisson_deviance_total(train["ClaimNb"], train["glm_pred_claim_count"]) / len(train),
        poisson_deviance_total(train["ClaimNb"], train["gbm_pred_claim_count"]) / len(train),
    ],
    "test_avg_poisson_deviance": [
        poisson_deviance_total(test["ClaimNb"], test["glm_pred_claim_count"]) / len(test),
        poisson_deviance_total(test["ClaimNb"], test["gbm_pred_claim_count"]) / len(test),
    ],
    "test_predicted_claim_count": [
        test["glm_pred_claim_count"].sum(),
        test["gbm_pred_claim_count"].sum(),
    ],
    "test_actual_to_expected_claims": [
        test["ClaimNb"].sum() / test["glm_pred_claim_count"].sum(),
        test["ClaimNb"].sum() / test["gbm_pred_claim_count"].sum(),
    ],
})
ml_frequency_metrics["overfit_gap_ratio"] = (
    ml_frequency_metrics["test_avg_poisson_deviance"]
    / ml_frequency_metrics["train_avg_poisson_deviance"]
    - 1
)
display(ml_frequency_metrics)


In [ ]:
def make_sev_gbm(params):
    return Pipeline(steps=[
        ("preprocess", make_preprocess()),
        ("model", HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=params["learning_rate"],
            max_iter=500,
            max_leaf_nodes=params["max_leaf_nodes"],
            min_samples_leaf=params["min_samples_leaf"],
            l2_regularization=params["l2_regularization"],
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=25,
            random_state=RANDOM_STATE,
        )),
    ])


severity_candidates = [
    {
        "candidate": "conservative",
        "learning_rate": 0.04,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 300,
        "l2_regularization": 1.00,
    },
    {
        "candidate": "balanced",
        "learning_rate": 0.05,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 150,
        "l2_regularization": 0.30,
    },
    {
        "candidate": "flexible",
        "learning_rate": 0.05,
        "max_leaf_nodes": 63,
        "min_samples_leaf": 75,
        "l2_regularization": 0.10,
    },
]

severity_models = {}
severity_rows = []

for params in severity_candidates:
    model = make_sev_gbm(params)
    model.fit(sev_train[model_features], sev_train["capped_claim_amount"])

    train_pred = np.maximum(model.predict(sev_train[model_features]), 1e-12)
    test_pred = np.maximum(model.predict(sev_test[model_features]), 1e-12)
    train_mae = mean_absolute_error(sev_train["capped_claim_amount"], train_pred)
    test_mae = mean_absolute_error(sev_test["capped_claim_amount"], test_pred)

    severity_models[params["candidate"]] = model
    severity_rows.append({
        "candidate": params["candidate"],
        "train_mae": train_mae,
        "test_mae": test_mae,
        "overfit_gap_ratio": test_mae / train_mae - 1,
        "train_rmse": rmse(sev_train["capped_claim_amount"], train_pred),
        "test_rmse": rmse(sev_test["capped_claim_amount"], test_pred),
        "n_iterations": getattr(model.named_steps["model"], "n_iter_", np.nan),
        **{k: v for k, v in params.items() if k != "candidate"},
    })

severity_tuning = pd.DataFrame(severity_rows).sort_values(["test_mae", "overfit_gap_ratio"])
display(severity_tuning)

eligible_severity = severity_tuning.loc[severity_tuning["overfit_gap_ratio"] <= 0.20]
if eligible_severity.empty:
    eligible_severity = severity_tuning
best_severity_name = eligible_severity.iloc[0]["candidate"]
sev_gbm = severity_models[best_severity_name]

train["gbm_pred_severity"] = np.maximum(sev_gbm.predict(train[model_features]), 1e-12)
test["gbm_pred_severity"] = np.maximum(sev_gbm.predict(test[model_features]), 1e-12)
train["gbm_pred_pure_premium"] = train["gbm_pred_frequency"] * train["gbm_pred_severity"]
test["gbm_pred_pure_premium"] = test["gbm_pred_frequency"] * test["gbm_pred_severity"]
train["gbm_expected_loss"] = train["gbm_pred_pure_premium"] * train["Exposure"]
test["gbm_expected_loss"] = test["gbm_pred_pure_premium"] * test["Exposure"]

model_comparison = pd.DataFrame({
    "model": [
        "GLM frequency x Gamma severity",
        f"GBM frequency ({best_freq_name}) x GBM severity ({best_severity_name})",
    ],
    "test_actual_loss": [test["total_claim_amount"].sum(), test["total_claim_amount"].sum()],
    "test_expected_loss": [test["glm_expected_loss"].sum(), test["gbm_expected_loss"].sum()],
    "test_actual_to_expected": [
        test["total_claim_amount"].sum() / test["glm_expected_loss"].sum(),
        test["total_claim_amount"].sum() / test["gbm_expected_loss"].sum(),
    ],
    "test_expected_pure_premium": [
        test["glm_expected_loss"].sum() / test["Exposure"].sum(),
        test["gbm_expected_loss"].sum() / test["Exposure"].sum(),
    ],
})
model_comparison["absolute_calibration_error"] = (
    model_comparison["test_actual_to_expected"] - 1
).abs()
display(model_comparison)


## 11. Variability and Overfit Diagnostics

A single point estimate can look precise when the underlying loss experience
is volatile. This section adds bootstrap confidence intervals and explicit
train/test gap checks. The goal is to improve accuracy without selecting a
model that only fits the training data.


In [ ]:
def bootstrap_portfolio_metrics(data, expected_loss_col, label, n_boot=300, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(data)
    actual_loss = data["total_claim_amount"].to_numpy()
    expected_loss = data[expected_loss_col].to_numpy()
    exposure = data["Exposure"].to_numpy()

    rows = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        actual = actual_loss[idx].sum()
        expected = expected_loss[idx].sum()
        exp = exposure[idx].sum()
        rows.append({
            "actual_to_expected": actual / expected,
            "actual_pure_premium": actual / exp,
            "expected_pure_premium": expected / exp,
        })

    boot = pd.DataFrame(rows)
    summary = boot.quantile([0.025, 0.50, 0.975]).T
    summary.columns = ["ci_025", "median", "ci_975"]
    summary.insert(0, "model", label)
    summary.insert(1, "metric", summary.index)
    return summary.reset_index(drop=True)


portfolio_variability = pd.concat(
    [
        bootstrap_portfolio_metrics(
            test, "glm_expected_loss", "GLM frequency x Gamma severity", seed=RANDOM_STATE
        ),
        bootstrap_portfolio_metrics(
            test,
            "gbm_expected_loss",
            f"GBM frequency ({best_freq_name}) x GBM severity ({best_severity_name})",
            seed=RANDOM_STATE + 1,
        ),
    ],
    ignore_index=True,
)
display(portfolio_variability)

glm_sev_train_mae = severity_metrics.loc[severity_metrics["split"] == "train", "mae_capped"].iloc[0]
glm_sev_test_mae = severity_metrics.loc[severity_metrics["split"] == "test", "mae_capped"].iloc[0]
selected_sev_row = severity_tuning.loc[severity_tuning["candidate"] == best_severity_name].iloc[0]

overfit_diagnostics = pd.concat(
    [
        ml_frequency_metrics.assign(component="frequency", metric="average Poisson deviance")[
            ["component", "model", "metric", "train_avg_poisson_deviance", "test_avg_poisson_deviance", "overfit_gap_ratio"]
        ].rename(
            columns={
                "train_avg_poisson_deviance": "train_metric",
                "test_avg_poisson_deviance": "test_metric",
            }
        ),
        pd.DataFrame({
            "component": ["severity", "severity"],
            "model": ["Gamma GLM", f"GBM severity ({best_severity_name})"],
            "metric": ["MAE on capped claims", "MAE on capped claims"],
            "train_metric": [glm_sev_train_mae, selected_sev_row["train_mae"]],
            "test_metric": [glm_sev_test_mae, selected_sev_row["test_mae"]],
            "overfit_gap_ratio": [
                glm_sev_test_mae / glm_sev_train_mae - 1,
                selected_sev_row["overfit_gap_ratio"],
            ],
        }),
    ],
    ignore_index=True,
)
display(overfit_diagnostics)


## 12. Lift and Calibration

Decile analysis groups policies by predicted pure premium and compares actual
versus expected loss costs.


In [ ]:
def lift_table(data, score_col, expected_loss_col, n_bins=10):
    out = data.copy()
    out["score_rank"] = out[score_col].rank(method="first")
    out["decile"] = pd.qcut(out["score_rank"], q=n_bins, labels=False) + 1
    table = (
        out.groupby("decile")
        .agg(
            policies=("IDpol", "count"),
            exposure=("Exposure", "sum"),
            actual_loss=("total_claim_amount", "sum"),
            expected_loss=(expected_loss_col, "sum"),
            claim_count=("ClaimNb", "sum"),
            average_score=(score_col, "mean"),
        )
        .reset_index()
    )
    table["actual_pure_premium"] = table["actual_loss"] / table["exposure"]
    table["expected_pure_premium"] = table["expected_loss"] / table["exposure"]
    table["actual_to_expected"] = table["actual_loss"] / table["expected_loss"]
    return table

glm_lift = lift_table(test, "glm_pred_pure_premium", "glm_expected_loss")
gbm_lift = lift_table(test, "gbm_pred_pure_premium", "gbm_expected_loss")

display(glm_lift)
display(gbm_lift)


In [ ]:
def plot_lift(table, title):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    x = table["decile"].astype(str)
    ax.plot(x, table["actual_pure_premium"], marker="o", linewidth=2, label="Actual")
    ax.plot(x, table["expected_pure_premium"], marker="s", linewidth=2, label="Expected")
    ax.set_xlabel("Predicted risk decile, low to high")
    ax.set_ylabel("Pure premium")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    return fig

fig_glm_lift = plot_lift(glm_lift, "GLM Actual vs Expected Pure Premium by Decile")
plt.show()

fig_gbm_lift = plot_lift(gbm_lift, "GBM Actual vs Expected Pure Premium by Decile")
plt.show()


## 13. Rating Relativities

This section compares observed and modeled relativities. Observed relativities
can be noisy in low-credibility cells. Modeled relativities smooth the raw
experience and are more suitable as pricing indications. Bootstrap intervals
are added for the key segment relativities to show variability.


In [ ]:
def relativity_table(data, group_col, expected_loss_col="glm_expected_loss"):
    summary = rating_summary(data, group_col, expected_loss_col=expected_loss_col)
    total_observed_pp = summary["paid_loss"].sum() / summary["exposure"].sum()
    total_modeled_pp = summary["expected_loss"].sum() / summary["exposure"].sum()
    summary["observed_relativity"] = summary["pure_premium"] / total_observed_pp
    summary["modeled_relativity"] = summary["modeled_pure_premium"] / total_modeled_pp
    return summary

relativity_outputs = {
    "driver_age": relativity_table(test, "DrivAgeBand"),
    "bonus_malus": relativity_table(test, "BonusMalusBand"),
    "vehicle_age": relativity_table(test, "VehAgeBand"),
    "area": relativity_table(test, "Area"),
    "region": relativity_table(test, "Region"),
}

def bootstrap_relativity_ci(data, group_col, expected_loss_col="glm_expected_loss", n_boot=250, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    groups = list(data[group_col].dropna().astype(str).unique())
    n = len(data)
    records = {group: {"observed_relativity": [], "actual_to_expected": []} for group in groups}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot = data.iloc[idx].copy()
        boot[group_col] = boot[group_col].astype(str)
        total_pp = boot["total_claim_amount"].sum() / boot["Exposure"].sum()
        grouped = (
            boot.groupby(group_col, observed=True)
            .agg(
                exposure=("Exposure", "sum"),
                paid_loss=("total_claim_amount", "sum"),
                expected_loss=(expected_loss_col, "sum"),
            )
            .reset_index()
        )
        grouped["observed_relativity"] = (grouped["paid_loss"] / grouped["exposure"]) / total_pp
        grouped["actual_to_expected"] = grouped["paid_loss"] / grouped["expected_loss"]

        for _, row in grouped.iterrows():
            group = str(row[group_col])
            if group in records:
                records[group]["observed_relativity"].append(row["observed_relativity"])
                records[group]["actual_to_expected"].append(row["actual_to_expected"])

    rows = []
    for group, metrics in records.items():
        row = {group_col: group}
        for metric, values in metrics.items():
            values = np.asarray(values, dtype=float)
            row[f"{metric}_ci_025"] = np.nanquantile(values, 0.025)
            row[f"{metric}_ci_500"] = np.nanquantile(values, 0.500)
            row[f"{metric}_ci_975"] = np.nanquantile(values, 0.975)
        rows.append(row)
    return pd.DataFrame(rows)

driver_age_variability = bootstrap_relativity_ci(test, "DrivAgeBand", seed=RANDOM_STATE + 10)
bonus_malus_variability = bootstrap_relativity_ci(test, "BonusMalusBand", seed=RANDOM_STATE + 11)

for key, group_col, ci_table in [
    ("driver_age", "DrivAgeBand", driver_age_variability),
    ("bonus_malus", "BonusMalusBand", bonus_malus_variability),
]:
    base = relativity_outputs[key].copy()
    base[group_col] = base[group_col].astype(str)
    relativity_outputs[key] = base.merge(ci_table, on=group_col, how="left")

display(relativity_outputs["driver_age"])
display(relativity_outputs["bonus_malus"])


In [ ]:
def plot_relativity(table, group_col, title):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    labels = table[group_col].astype(str).tolist()
    x = np.arange(len(labels))
    if {"observed_relativity_ci_025", "observed_relativity_ci_975"}.issubset(table.columns):
        ax.fill_between(
            x,
            table["observed_relativity_ci_025"],
            table["observed_relativity_ci_975"],
            color="#9ecae1",
            alpha=0.35,
            label="Observed 95% bootstrap interval",
        )
    ax.plot(x, table["observed_relativity"], marker="o", linewidth=2, label="Observed")
    ax.plot(x, table["modeled_relativity"], marker="s", linewidth=2, label="Modeled GLM")
    ax.axhline(1.0, color="black", linewidth=1, linestyle="--")
    ax.set_ylabel("Relativity to portfolio average")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.legend()
    fig.tight_layout()
    return fig

fig_driver_age_rel = plot_relativity(
    relativity_outputs["driver_age"], "DrivAgeBand", "Driver Age Relativities"
)
plt.show()

fig_bonus_rel = plot_relativity(
    relativity_outputs["bonus_malus"], "BonusMalusBand", "Bonus-Malus Relativities"
)
plt.show()


## 14. Model Interpretability

GLM coefficients provide direct directional interpretation. For the machine
learning benchmark, permutation importance gives a model-agnostic view of
key predictors.


In [ ]:
def coefficient_table(result, top_n=20):
    coefs = pd.DataFrame({
        "term": result.params.index,
        "coef": result.params.values,
        "relativity": np.exp(result.params.values),
        "p_value": result.pvalues.values,
    })
    coefs["abs_log_relativity"] = np.abs(coefs["coef"])
    return coefs.sort_values("abs_log_relativity", ascending=False).head(top_n)

freq_coef = coefficient_table(freq_glm, top_n=25)
sev_coef = coefficient_table(sev_glm, top_n=25)

display(freq_coef)
display(sev_coef)


In [ ]:
importance_sample = test.sample(n=min(20000, len(test)), random_state=RANDOM_STATE)

freq_importance = permutation_importance(
    freq_gbm,
    importance_sample[model_features],
    importance_sample["ClaimNb"] / importance_sample["Exposure"],
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error",
)

importance_table = pd.DataFrame({
    "feature": model_features,
    "importance_mean": freq_importance.importances_mean,
    "importance_std": freq_importance.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance_table)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=importance_table.head(12),
    x="importance_mean",
    y="feature",
    ax=ax,
    color="#3182bd",
)
ax.set_title("GBM Frequency Model Permutation Importance")
ax.set_xlabel("Increase in MAE when permuted")
ax.set_ylabel("")
fig.tight_layout()
plt.show()


## 15. Detailed Result Interpretation

This cell turns the numeric outputs into a concise actuarial interpretation.
It is intentionally detailed so the analysis can stand on its own as a
project report.


In [ ]:
def fmt_num(x, digits=2):
    return f"{x:,.{digits}f}"


def fmt_pct(x, digits=1):
    return f"{100 * x:,.{digits}f}%"


glm_freq_row = ml_frequency_metrics.loc[ml_frequency_metrics["model"].str.contains("GLM")].iloc[0]
gbm_freq_row = ml_frequency_metrics.loc[ml_frequency_metrics["model"].str.contains("GBM")].iloc[0]
glm_pp_row = model_comparison.iloc[0]
gbm_pp_row = model_comparison.iloc[1]

selected_prediction_row = model_comparison.sort_values("absolute_calibration_error").iloc[0]
selected_prediction_model = selected_prediction_row["model"]

glm_ae_ci = portfolio_variability.loc[
    (portfolio_variability["model"].str.contains("GLM"))
    & (portfolio_variability["metric"] == "actual_to_expected")
].iloc[0]
gbm_ae_ci = portfolio_variability.loc[
    (portfolio_variability["model"].str.contains("GBM"))
    & (portfolio_variability["metric"] == "actual_to_expected")
].iloc[0]

freq_improvement = (
    glm_freq_row["test_avg_poisson_deviance"] - gbm_freq_row["test_avg_poisson_deviance"]
) / glm_freq_row["test_avg_poisson_deviance"]

top_frequency_features = ", ".join(importance_table.head(5)["feature"].tolist())

analysis_report_lines = [
    "Detailed actuarial interpretation",
    "",
    f"1. Data scale: the modeling file contains {len(pricing):,} policy records, "
    f"{pricing['Exposure'].sum():,.0f} exposure units, {pricing['ClaimNb'].sum():,.0f} reported claims, "
    f"and total paid loss of {pricing['total_claim_amount'].sum():,.0f}.",
    "",
    f"2. Frequency model: the GLM test average Poisson deviance is "
    f"{fmt_num(glm_freq_row['test_avg_poisson_deviance'], 4)}, while the selected GBM is "
    f"{fmt_num(gbm_freq_row['test_avg_poisson_deviance'], 4)}. "
    f"The relative improvement is {fmt_pct(freq_improvement, 2)}. "
    f"The GBM train/test gap is {fmt_pct(gbm_freq_row['overfit_gap_ratio'], 2)}, "
    "which is reviewed as an overfitting control rather than accepting the lowest training error.",
    "",
    f"3. Severity model: the Gamma GLM provides an interpretable baseline on capped claims. "
    f"The selected severity GBM is '{best_severity_name}', with test MAE "
    f"{fmt_num(selected_sev_row['test_mae'], 2)} and train/test gap "
    f"{fmt_pct(selected_sev_row['overfit_gap_ratio'], 2)}. "
    "The cap at the 99.5th percentile reduces distortion from extreme claims while preserving "
    "large-loss diagnostics for actuarial judgment.",
    "",
    f"4. Pure premium calibration: GLM actual-to-expected on the test set is "
    f"{fmt_num(glm_pp_row['test_actual_to_expected'], 3)} with bootstrap 95% interval "
    f"[{fmt_num(glm_ae_ci['ci_025'], 3)}, {fmt_num(glm_ae_ci['ci_975'], 3)}]. "
    f"GBM actual-to-expected is {fmt_num(gbm_pp_row['test_actual_to_expected'], 3)} with interval "
    f"[{fmt_num(gbm_ae_ci['ci_025'], 3)}, {fmt_num(gbm_ae_ci['ci_975'], 3)}].",
    "",
    f"5. Model selection view: based on absolute calibration error, the strongest predictive candidate is "
    f"'{selected_prediction_model}'. For a rate filing or pricing review, the GLM remains the primary "
    "actuarial benchmark because it produces stable, explainable relativities. The GBM is used as a "
    "challenger model to test non-linear segmentation and potential lift.",
    "",
    f"6. Key drivers: the GBM frequency importance ranking highlights {top_frequency_features}. "
    "These factors should be compared with GLM coefficient relativities and segment credibility "
    "before turning model output into rate indications.",
    "",
    "7. Variability: bootstrap intervals around portfolio A/E and segment relativities are included because "
    "auto claim severity is skewed. Segments with wide intervals should receive less credibility or be "
    "combined before implementation.",
]

analysis_report_text = "\n".join(analysis_report_lines)
print(analysis_report_text)


## 16. Actuarial Judgment Notes

**Credibility:** Raw observed pure premiums are volatile in small cells,
especially for regions or rating bands with low exposure. Modeled relativities
should be reviewed alongside exposure volume and claim counts.

**Large losses:** The severity model caps individual claims at the 99.5th
percentile. A production rate indication would pair the capped model with a
separate large-loss or excess-layer provision.

**Model selection:** The GLM is the preferred pricing baseline because it
provides transparent relativities, clear exposure treatment, and easier
actuarial review. The GBM benchmark tests whether non-linear interactions
materially improve segmentation.

**Fairness and regulation:** Driver age and geography can be predictive, but
require business and regulatory review before rate implementation.

**Limitations:** The project uses public data and does not include trend,
expenses, profit, reinsurance, or regulatory constraints. Results are pure
premium indications rather than final manual rates.


## 17. Save Project Outputs

The final cell exports tables and charts that support reproducibility and
make the analysis easier to review outside the notebook.


In [ ]:
frequency_metrics.to_csv(OUTPUT_DIR / "frequency_glm_metrics.csv", index=False)
severity_metrics.to_csv(OUTPUT_DIR / "severity_glm_metrics.csv", index=False)
pure_premium_metrics.to_csv(OUTPUT_DIR / "pure_premium_glm_metrics.csv", index=False)
freq_tuning.to_csv(OUTPUT_DIR / "frequency_gbm_tuning.csv", index=False)
severity_tuning.to_csv(OUTPUT_DIR / "severity_gbm_tuning.csv", index=False)
model_comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
portfolio_variability.to_csv(OUTPUT_DIR / "portfolio_bootstrap_variability.csv", index=False)
overfit_diagnostics.to_csv(OUTPUT_DIR / "overfit_diagnostics.csv", index=False)
glm_lift.to_csv(OUTPUT_DIR / "glm_lift_deciles.csv", index=False)
gbm_lift.to_csv(OUTPUT_DIR / "gbm_lift_deciles.csv", index=False)
importance_table.to_csv(OUTPUT_DIR / "gbm_frequency_feature_importance.csv", index=False)
driver_age_variability.to_csv(OUTPUT_DIR / "driver_age_bootstrap_relativity_ci.csv", index=False)
bonus_malus_variability.to_csv(OUTPUT_DIR / "bonus_malus_bootstrap_relativity_ci.csv", index=False)

for name, table in relativity_outputs.items():
    table.to_csv(OUTPUT_DIR / f"relativity_{name}.csv", index=False)

(OUTPUT_DIR / "actuarial_interpretation.txt").write_text(
    analysis_report_text,
    encoding="utf-8",
)

fig_glm_lift.savefig(OUTPUT_DIR / "glm_actual_vs_expected_lift.png", dpi=160, bbox_inches="tight")
fig_gbm_lift.savefig(OUTPUT_DIR / "gbm_actual_vs_expected_lift.png", dpi=160, bbox_inches="tight")
fig_driver_age_rel.savefig(OUTPUT_DIR / "driver_age_relativity.png", dpi=160, bbox_inches="tight")
fig_bonus_rel.savefig(OUTPUT_DIR / "bonus_malus_relativity.png", dpi=160, bbox_inches="tight")

print(f"Saved project outputs to: {OUTPUT_DIR}")


## Conclusion

This project estimates **auto insurance pure premium**, which is the expected
claim cost per unit of exposure. It is the core actuarial component of a
technical price, but it is not the final customer premium. A final premium
would also require expense loads, profit provision, trend, reinsurance,
taxes/fees, and regulatory or business constraints.

On the test set, the observed loss cost is **169.99** per exposure unit. The
selected GBM pricing model estimates an expected pure premium of **169.88**,
producing an actual-to-expected loss ratio of **1.001**. The GLM baseline
estimates **169.61**, with an actual-to-expected ratio of **1.002**. Both
models are closely calibrated at the portfolio level, with the GBM providing
the slightly better point calibration.

The GBM challenger also improves frequency model fit: test average Poisson
deviance decreases from **0.3186** for the GLM to **0.3032** for the selected
GBM, while the train/test gap remains modest at about **4.9%**. For severity,
the selected conservative GBM has a capped-claim test MAE of **1,320.94**,
slightly better than the Gamma GLM test MAE of **1,325.49**. These results
indicate a small but measurable predictive improvement without obvious
overfitting on the selected split.

For pricing use, the GLM remains an important benchmark because it produces
transparent rating relativities that are easier to review and explain. The
GBM is useful as a challenger model and can identify non-linear segmentation
that may not be fully captured by the GLM.

The main caveat is variability. The bootstrap 95% interval for portfolio
actual-to-expected loss ratio is approximately **[0.859, 1.157]**, reflecting
the volatility of claim severity. Therefore, even though the point estimate is
well calibrated, final pricing indications should still apply credibility
judgment, large-loss review, and segment-level reasonableness checks before
being used as final rate recommendations.
